# ANPR Plate Detector Training — Google Colab

This notebook trains the **plate detector**. Do not run the permanent parking database/API in Colab; Colab runtimes are temporary.

Permanent flow: **train here → download `best.pt` → put it on your permanent Docker server**.


## 0. Prepare your dataset

Use YOLO object-detection format with one class named `plate`:
```text
plate_dataset/
  data.yaml
  images/train/
  images/val/
  labels/train/
  labels/val/
```
Each label row is: `class_id x_center y_center width height` using normalized coordinates.


In [ ]:
!nvidia-smi
!pip install -q -U ultralytics


In [ ]:
import torch
from ultralytics import YOLO
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 1. Upload the YOLO dataset ZIP


In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded))
print('Dataset:', zip_name)


In [ ]:
from pathlib import Path
import zipfile, shutil

extract_root = Path('/content/anpr_data')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_root)

yaml_files = list(extract_root.rglob('data.yaml'))
assert yaml_files, 'data.yaml was not found.'
data_yaml = str(yaml_files[0])
print('Using:', data_yaml)


In [ ]:
import yaml
dataset_root = Path(data_yaml).parent
with open(data_yaml, 'r') as f:
    cfg = yaml.safe_load(f)
cfg['path'] = str(dataset_root)
if 'names' not in cfg:
    cfg['names'] = {0: 'plate'}
with open(data_yaml, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('Patched data.yaml path to:', dataset_root)
print(cfg)


## 2. Check dataset counts


In [ ]:
root = Path(data_yaml).parent
for split in ['train','val','test']:
    img_dir = root / 'images' / split
    label_dir = root / 'labels' / split
    if img_dir.exists():
        print(split, 'images:', len(list(img_dir.iterdir())), 'labels:', len(list(label_dir.iterdir())) if label_dir.exists() else 0)


## 3. Train the detector

This uses a small pretrained YOLO26 checkpoint as a current fine-tuning baseline. If your installed Ultralytics release does not expose `yolo26n.pt`, replace it with a supported small detection checkpoint in that release.


In [ ]:
model = YOLO('yolo26n.pt')
results = model.train(
    data=data_yaml,
    epochs=80,
    imgsz=640,
    batch=16,
    patience=20,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/content/anpr_runs',
    name='malaysia_plate_detector',
    exist_ok=True,
    plots=True,
)


## 4. Validate the trained model


In [ ]:
best_path = '/content/anpr_runs/malaysia_plate_detector/weights/best.pt'
best_model = YOLO(best_path)
metrics = best_model.val(data=data_yaml)
print(metrics)


## 5. Test with a real vehicle image


In [ ]:
test_upload = files.upload()
test_image = next(iter(test_upload))
best_model.predict(
    source=test_image,
    conf=0.35,
    save=True,
    project='/content/anpr_test',
    name='prediction',
    exist_ok=True,
)
print('Saved to /content/anpr_test/prediction')


In [ ]:
from IPython.display import display
from PIL import Image
out = Path('/content/anpr_test/prediction')
for p in list(out.glob('*'))[:5]:
    if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}:
        display(Image.open(p))


## 6. Download `best.pt`

After download, copy it to `backend/models/best.pt` in the permanent project and restart the backend.


In [ ]:
files.download(best_path)
